# 01 - Whole Slide Image (WSI) Data Extraction

## Notebook Objective
Due to the massive file size of Whole Slide Images (`.svs` files, often exceeding several gigabytes and billions of pixels), direct ingestion into neural network architectures is computationally unfeasible. 

The primary objective of this notebook is to process the raw WSI dataset by mapping the provided `.xml` annotations (Ground Truth spatial coordinates) to the high-resolution images. This script will programmatically calculate bounding boxes with contextual margins around the irregular polygonal annotations, extract the specific tissue patches at maximum magnification (Level 0), and save them as individual `.png` files. 

This process transforms the unmanageable gigapixel WSIs into an optimized, discrete dataset of isolated glomeruli images, preparing the pipeline for the subsequent Unsupervised Machine Learning phase (Clustering).

---

### Setup and WSI Loading

In [ ]:
import openslide
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import numpy as np

# Define file paths
wsi_path = "./glomeruli_grading/RECHERCHE-003.svs"   
xml_path = "./glomeruli_grading/RECHERCHE-003.xml"

# Load the WSI into memory (execute only once)
print("Loading WSI...")
slide = openslide.OpenSlide(wsi_path)
print(f"WSI loaded. Dimensions: {slide.dimensions} pixels")

### Background Extraction

In [ ]:
# Select coordinates outside the tissue (background)
x_bg, y_bg = 5000, 5000 
patch_size = (512, 512)

# Extract background patch
patch_bg = slide.read_region((x_bg, y_bg), 0, patch_size).convert("RGB")

# Visualize background patch
plt.figure(figsize=(4, 4))
plt.imshow(patch_bg)
plt.title(f"Negative Sample (Background)\nX={x_bg}, Y={y_bg}")
plt.axis("off")
plt.show()

### First Glomerulus Extraction

In [ ]:
# Parse the XML file
tree = ET.parse(xml_path)
root = tree.getroot()

# Get the first annotation
first_annotation = root.find('.//Annotation')
annotation_name = first_annotation.get('Name')

# Extract X and Y coordinates
x_coords = [float(coord.get('X')) for coord in first_annotation.findall('.//Coordinate')]
y_coords = [float(coord.get('Y')) for coord in first_annotation.findall('.//Coordinate')]

# Calculate the Bounding Box with a safety margin
margin = 50
start_x = int(min(x_coords)) - margin
start_y = int(min(y_coords)) - margin
width = int(max(x_coords) - min(x_coords)) + (margin * 2)
height = int(max(y_coords) - min(y_coords)) + (margin * 2)

print(f"Bounding Box Coordinates: Start X={start_x}, Start Y={start_y}, Dim={width}x{height}")

# Extract glomerulus patch
glomerulus_patch = slide.read_region((start_x, start_y), 0, (width, height)).convert("RGB")

# Visualize positive patch
plt.figure(figsize=(6, 6))
plt.imshow(glomerulus_patch)
plt.title(f"Positive Sample: {annotation_name}")
plt.axis("off")
plt.show()

### Dataset creation

In [ ]:
import os
import openslide
import xml.etree.ElementTree as ET

# 1. Define dataset and output directories
dataset_dir = "./glomeruli_grading/"
save_dir = "./dataset_glomeruli_all"

# Create the master save directory
os.makedirs(save_dir, exist_ok=True)
print(f"Master save directory ready: {save_dir}\n")

total_saved_count = 0

# 2. Iterate through all files in the dataset directory
for file_name in os.listdir(dataset_dir):
    
    # Check if the file is a Whole Slide Image
    if file_name.endswith(".svs"):
        
        # Construct paths for SVS and the expected XML
        wsi_path = os.path.join(dataset_dir, file_name)
        
        # Replace .svs extension with .xml
        xml_name = file_name.replace(".svs", ".xml")
        xml_path = os.path.join(dataset_dir, xml_name)
        
        # Extract the base name (e.g., "RECHERCHE-003") for naming the patches
        base_wsi_name = file_name.replace(".svs", "")
        
        # 3. Check if the corresponding XML exists
        if not os.path.exists(xml_path):
            print(f"WARNING: XML not found for {file_name}. Skipping...")
            continue
            
        print(f"Processing WSI: {file_name}")
        
        # Load WSI and parse XML
        try:
            slide = openslide.OpenSlide(wsi_path)
            tree = ET.parse(xml_path)
            root = tree.getroot()
        except Exception as e:
            print(f"ERROR reading files for {file_name}: {e}. Skipping...")
            continue
            
        all_annotations = root.findall('.//Annotation')
        wsi_saved_count = 0
        
        # 4. Inner loop: Extract annotations for the current WSI
        for annotation in all_annotations:
            annotation_name = annotation.get('Name')
            coordinates = annotation.findall('.//Coordinate')
            
            # Safety check
            if len(coordinates) < 3:
                continue
                
            x_coords = [float(coord.get('X')) for coord in coordinates]
            y_coords = [float(coord.get('Y')) for coord in coordinates]
            
            # Bounding Box calculation
            margin = 50
            start_x = int(min(x_coords)) - margin
            start_y = int(min(y_coords)) - margin
            width = int(max(x_coords) - min(x_coords)) + (margin * 2)
            height = int(max(y_coords) - min(y_coords)) + (margin * 2)
            
            # Patch extraction
            try:
                patch = slide.read_region((start_x, start_y), 0, (width, height)).convert("RGB")
            except Exception as e:
                print(f"  -> ERROR extracting patch {annotation_name}: {e}")
                continue
            
            # 5. Save the patch with a unique name
            # Format: WSI-Name_Annotation-Name.png (e.g., RECHERCHE-003_Annotation_0.png)
            clean_annotation_name = annotation_name.replace(" ", "_")
            final_file_name = f"{base_wsi_name}_{clean_annotation_name}.png"
            file_path = os.path.join(save_dir, final_file_name)
            
            patch.save(file_path)
            wsi_saved_count += 1
            total_saved_count += 1
            
        print(f"  -> Extracted {wsi_saved_count} patches from {file_name}.")
        
        # Close the slide to free up memory
        slide.close()

print(f"\nPROCESS COMPLETE! A total of {total_saved_count} glomeruli were extracted across the entire dataset.")